# Movie Recommendations System

### Part 1 - Imports

In [158]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Part 2 - Load and explore the data

In [ ]:
dir = "ml-latest//"
movies = pd.read_csv(dir + "movies.csv")
tags = pd.read_csv(dir + "tags.csv")
tags = tags.sample(n=500000, random_state=42)


n = 5 #This will be the number of suggestions we will see in the end result 
movie_title = "Jumanji" #This will be the movie that will compare with the rest



### Explore and Clean Movies

In [ ]:
#print(f"Movies : \n {movies.head()}")
#print(f"\n Ratings :\n {ratings.head()}")
#print(f"\n Tags : \n {tags.head()}")
#print(f"Movies Nulls: \n{movies.info()}")
# We check how many movies don't have a genre assigned
#movies[movies["genres"] == "(no genres listed)"].shape[0]

# and now we remove those movies from the list as they won't be useful for the recommendation based on genres
movies = movies[movies["genres"] != "(no genres listed)"].reset_index(drop=True)

#I'll use regex to remove the years from the title and instead add them to a new column so they can be used for advanced filtering.
#This means in case of duplicates we'll have to ask the user to choose between several options
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)$")
#This part extracted the part in parentheses containing 4 digits, and added it to a new column
movies["title"] = movies["title"].str.replace(r"\(\d{4}\)$", "", regex=True).str.strip()
#This line removes the years from the title with the same parameters as the previous
#regex=True tells pandas to interpret \(\d{4}\)$ as pattern instead of literal string of text
#str.strip() removes any leftover whitespace after removal

# since the genres in the database are divided by | and pandas expects a space we will adapt them
movies["genres"] = movies["genres"].str.replace("|", " ", regex=False)

<class 'pandas.DataFrame'>
RangeIndex: 86537 entries, 0 to 86536
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  86537 non-null  int64
 1   title    86537 non-null  str  
 2   genres   86537 non-null  str  
dtypes: int64(1), str(2)
memory usage: 2.0 MB
Movies Nulls: 
None
Movies sample: 
0    Adventure Animation Children Comedy Fantasy
1                     Adventure Children Fantasy
2                                 Comedy Romance
3                           Comedy Drama Romance
4                                         Comedy
Name: genres, dtype: str


### Explore and Clean Tags

In [ ]:
print(f"Tags sample:\n{tags.head()}")
print(f"Tags info:\n{tags.info()}")

#We can see there are two columns we have no use for so we can remove them: UserID and timestamp
tags = tags.drop(columns=["userId","timestamp"])

Tags sample:
         userId  movieId         tag   timestamp
178115    36119    26034         TCM  1650392412
1102583  215490     1274     yelling  1629171061
558328   109478   171763      action  1522781876
311770    62130    60069  Animation   1638238972
205962    43153   107406    Allegory  1517136311
<class 'pandas.DataFrame'>
Index: 500000 entries, 178115 to 12200
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   userId     500000 non-null  int64
 1   movieId    500000 non-null  int64
 2   tag        499996 non-null  str  
 3   timestamp  500000 non-null  int64
dtypes: int64(3), str(1)
memory usage: 19.1 MB
Tags info:
None


In [162]:
#Now that we only have the data we need, we can merge the rows with the same movieID to collect all tags in a single column
tags = tags.dropna(subset=["tag"])
tags_grouped = tags.groupby("movieId")["tag"].apply(lambda x: " ".join(x)).reset_index()
print(tags_grouped.head())

   movieId                                                tag
0        1  escape attempt funny animation clever fear fam...
1        2  shotgun construction site Lebbat Robin William...
2        3  fishing midwest CLV duringcreditsstinger old J...
3        4                                              slurs
4        5  father Steve Martin steve martin gynecologist ...


### Merging Dataframes
Now that we have both movies and tags clean and ready, we can merge them together

In [163]:
movies = movies.merge(tags_grouped, on="movieId",how="left")
#Now we have a column for genres and one for tags. We want to merge them 
#First we need to take care of the NaN values by replacing them with an empty string
movies["tag"] = movies["tag"].fillna("")
movies["combined"] = movies["genres"]+ " " + movies["tag"]

#Now that genres and tags are merged we can drop the tag column as it will be redundant. 
#We keep genres though since in a complete version one might want to look for movies by genre only
movies = movies.drop("tag", axis=1)
print(movies.head())

   movieId                        title  \
0        1                    Toy Story   
1        2                      Jumanji   
2        3             Grumpier Old Men   
3        4            Waiting to Exhale   
4        5  Father of the Bride Part II   

                                        genres  year  \
0  Adventure Animation Children Comedy Fantasy  1995   
1                   Adventure Children Fantasy  1995   
2                               Comedy Romance  1995   
3                         Comedy Drama Romance  1995   
4                                       Comedy  1995   

                                            combined  
0  Adventure Animation Children Comedy Fantasy es...  
1  Adventure Children Fantasy shotgun constructio...  
2  Comedy Romance fishing midwest CLV duringcredi...  
3                         Comedy Drama Romance slurs  
4  Comedy father Steve Martin steve martin gyneco...  


### Creating the TF-IDF vocabulary

In [164]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["combined"])

print(tfidf_matrix.shape)

(79477, 28909)


### Movie selection functions

In [165]:
def get_index (movie_title):
    result = movies[movies["title"]== movie_title]
    if result.empty:
        print(f"Movie {movie_title} not in the database")
        return None
    return result.index[0]

def get_vector(index, tfidf_matrix):
    row = tfidf_matrix[index].toarray()
    return row

def get_similarity (index, tfidf_matrix):
    row = get_vector(index, tfidf_matrix)
    similarity = cosine_similarity(row, tfidf_matrix)
    return similarity


### Recommend Movies

In [ ]:
def recommend_movies(movie_title, tfidf_matrix):
    idx = get_index(movie_title)
    if idx is None:
        return

    similarity = get_similarity(idx, tfidf_matrix)
    sim = similarity[0]

    sorted_sim = sim.argsort()[::-1]
    matches = []

#This system will make sure the title in our search is not included in the results even in case it's not in first position
    for idx in sorted_sim:
        if len(matches) == n:
            break
        if movies["title"].iloc[idx] != movie_title:
            matches.append(movies["title"].iloc[idx])

    return matches


### Running the model

In [167]:
print(recommend_movies(movie_title, tfidf_matrix))

['Flubber', 'Mrs. Doubtfire', "World's Greatest Dad", 'Final Cut, The', "A Merry Friggin' Christmas"]
